In [1]:
import sys
sys.path.append("..")
from src.preprocessing import run_full_preprocessing

data = run_full_preprocessing("../data/raw/wfp_food_prices_tza.csv")

mbeya_region = data["Mbeya"]
iringa_region = data["Iringa"]

print(mbeya_region.shape, iringa_region.shape)
print(mbeya_region.tail())
print(iringa_region.tail())

RUNNING FULL PREPROCESSING PIPELINE
[filter] total rows loaded: 58,845
[filter] matching commodity/pricetype/region (before unit filter): 385
[filter] excluded 4 row(s) with non-'100 KG' unit:
      date admin1       market unit  price
2011-10-15  Mbeya    Mwanjelwa   MT 453.26
2017-10-15  Mbeya    Mwanjelwa   MT 442.20
2018-03-15 Iringa Iringa Urban   MT 305.73
2018-09-15  Mbeya    Mwanjelwa   MT 295.04
[filter] final filtered rows (unit='100 KG'): 381
    Mbeya: 172 rows
    Iringa: 209 rows
[aggregate] Mbeya: 3 market(s) -> 154 months
[aggregate] Iringa: 1 market(s) -> 209 months
[reindex] shared calendar: 2006-01-01 to 2026-03-01 (243 months)
    Mbeya: 243 total, 154 observed, 89 missing (36.6%)
    Iringa: 243 total, 209 observed, 34 missing (14.0%)
PREPROCESSING COMPLETE
(243, 2) (243, 2)
          date    price
238 2025-11-01  57854.5
239 2025-12-01  60687.5
240 2026-01-01  67500.0
241 2026-02-01  70587.5
242 2026-03-01  67701.5
          date    price
238 2025-11-01  57818.0
2

In [2]:
# save the reindexed-but-not-yet-imputed checkpoint
mbeya_region.to_csv("../data/processed/mbeya_reindexed.csv", index=False)
iringa_region.to_csv("../data/processed/iringa_reindexed.csv", index=False)

print("saved checkpoint files to data/processed/")

saved checkpoint files to data/processed/


In [3]:
import sys
sys.path.append("..")

from src.imputation import impute_series

mbeya_clean = impute_series(mbeya_region, "Mbeya")
iringa_clean = impute_series(iringa_region, "Iringa")

=== Mbeya ===
before: 89 missing of 243
large gaps found (>= 4 months): 4
  2007-12-01 to 2008-03-01 (4 months)
    2008-01-01: previous year also missing, left unresolved
  2021-01-01 to 2021-04-01 (4 months)
    2021-03-01: previous year also missing, left unresolved
  2021-09-01 to 2022-04-01 (8 months)
    2022-03-01: previous year also missing, left unresolved
  2023-03-01 to 2023-07-01 (5 months)
    2023-03-01: previous year also missing, left unresolved
    2023-07-01: previous year also missing, left unresolved
after seasonal fill: 73 missing
linear interpolation filled: 71
status: 2 months still missing -- dropping (edge of series, no reference data available)
date
2006-01-01   NaN
2006-02-01   NaN
Name: price, dtype: float64

=== Iringa ===
before: 34 missing of 243
large gaps found (>= 4 months): 0
after seasonal fill: 34 missing
linear interpolation filled: 34
status: fully resolved



In [4]:
print(mbeya_clean.shape)
print(iringa_clean.shape)
print(mbeya_clean.isna().sum())
print(iringa_clean.isna().sum())

(241, 2)
(243, 2)
date     0
price    0
dtype: int64
date     0
price    0
dtype: int64


In [5]:
mbeya_clean.to_csv("../data/processed/mbeya_clean.csv", index=False)
iringa_clean.to_csv("../data/processed/iringa_clean.csv", index=False)

print("saved final cleaned files")


saved final cleaned files


In [6]:
from src.outliers import detect_outliers

mbeya_flagged, mbeya_outliers = detect_outliers(mbeya_clean, "Mbeya")
iringa_flagged, iringa_outliers = detect_outliers(iringa_clean, "Iringa")

=== Mbeya ===
Q1: 34889  Q3: 64000  IQR: 29111
bounds: [-52444, 151333]
no outliers flagged

=== Iringa ===
Q1: 32639  Q3: 59229  IQR: 26590
bounds: [-47130, 138998]
no outliers flagged

